# Підсумок усіх експериментів: таблиці та графіки для звіту
Запускається локально з `code/notebooks/`. Читає `code/results/runs/<RUN_ID>/metrics.json` і `results.csv` (скопійовані з Output
Kaggle-ноутбуків) та `code/results/experiments.csv`; картинки зберігає в `images/` репозиторію.

## 1. Метрики всіх ранів

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NAMES = ["person", "bicycle", "car", "van", "truck", "tricycle", "bus", "motor"]
RUNS = Path("../results/runs")
FIG = Path("../../images"); FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})
FAMILY = {"A": "anchor-free (YOLO11)", "T": "two-stage (Faster R-CNN)", "R": "one-stage anchor (RetinaNet)",
          "Q": "query-based (RT-DETR)", "F": "final pipeline"}
COL = {"A": "#2a78d6", "T": "#eb6834", "R": "#1baf7a", "Q": "#6250d6", "F": "#e34948"}

runs = {}
for p in sorted(RUNS.glob("*/metrics.json")):
    m = json.load(open(p))
    rid = m.get("run_id", p.parent.name)
    if m.get("debug"):
        continue
    runs[rid] = dict(m, _dir=p.parent)
main = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR500", "FPS"]
tab = pd.DataFrame({k: {c: v.get(c, np.nan) for c in main} for k, v in runs.items()}).T.sort_index()
tab.insert(0, "family", [FAMILY.get(k[0], "?") for k in tab.index])
display(tab.round(4))

## 2. Найкращий ран кожного сімейства: AP за розміром об'єктів

In [ ]:
best = tab.groupby("family").mAP.idxmax()
sub = tab.loc[best.values, ["AP_small", "AP_medium", "AP_large", "mAP"]]
display(sub.round(4))
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(3); w = 0.8 / len(sub)
for i, (rid, r) in enumerate(sub.iterrows()):
    ax.bar(x + (i - (len(sub) - 1) / 2) * w, r[["AP_small", "AP_medium", "AP_large"]].values, w,
           label=f"{rid} (mAP {r.mAP:.3f})", color=COL[rid[0]])
ax.set_xticks(x, ["small (<32²)", "medium", "large (>96²)"]); ax.set_ylabel("AP@[.5:.95]")
ax.set_title("Best run of each family: AP by object size (val)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "families_by_size.png", dpi=150); plt.show()

## 3. RT-DETR vs YOLO: val mAP по епохах при однаковому бюджеті (10 епох, 1024 px)
Криві з `results.csv` Ultralytics (mAP без ignore-регіонів, але однаково для обох моделей).

In [ ]:
def curve(rid):
    f = RUNS / rid / "results.csv"
    if not f.exists():
        return None
    d = pd.read_csv(f); d.columns = [c.strip() for c in d.columns]
    return d[["epoch", "metrics/mAP50-95(B)"]]

fig, ax = plt.subplots(figsize=(8, 3.8))
for rid, label in [("A0_yolo11s_1024", "YOLO11s (A0)"), ("A3_yolo11m_1280", "YOLO11m 1280 (A3)"),
                   ("Q0_rtdetr_l_1024", "RT-DETR-l (Q0)"), ("Q1_rtdetr_l_nq500_1024", "RT-DETR-l 500 q (Q1)")]:
    d = curve(rid)
    if d is not None:
        ax.plot(d.epoch, d["metrics/mAP50-95(B)"], "-o", lw=2, ms=5, label=label, color=COL[rid[0]],
                alpha=1.0 if rid in ("A0_yolo11s_1024", "Q0_rtdetr_l_1024") else 0.5)
ax.set_xlabel("epoch"); ax.set_ylabel("val mAP50-95 (Ultralytics)"); ax.set_title("Query-based vs CNN under the same budget")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIG / "detr_vs_yolo.png", dpi=150); plt.show()

## 4. AP по класах (найкращий ран кожного сімейства)

In [ ]:
pc = pd.DataFrame({rid: {n: runs[rid].get(f"AP_{n}", np.nan) for n in NAMES} for rid in best.values}).T
display(pc.round(3))
fig, ax = plt.subplots(figsize=(9, 0.6 * len(pc) + 1.2))
im = ax.imshow(pc.values, cmap="Blues", vmin=0, vmax=max(0.5, np.nanmax(pc.values)))
ax.set_xticks(range(len(NAMES)), NAMES, rotation=30); ax.set_yticks(range(len(pc)), pc.index); ax.grid(False)
for i in range(pc.shape[0]):
    for j in range(pc.shape[1]):
        ax.text(j, i, f"{pc.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="AP"); ax.set_title("Per-class AP (val)")
plt.tight_layout(); plt.savefig(FIG / "per_class_ap.png", dpi=150); plt.show()
print("картинки:", sorted(p.name for p in FIG.iterdir()))